In [1]:
# 计算 test_grid.ipynb 的“训练监督(render_3d) vs 输出图”RMSE
import os
import re
import glob
from pathlib import Path

import cv2
import numpy as np

GT_DIR = "/data/test_grid4_from_test_grid4/render_3d"
OUT_DIR = "/home/viplab/liukejia/project/mv_model/results_grid4"
TARGET_ANGLES = [0, 90, 180, 270]

def parse_name_tokens(stem: str):
    parts = stem.split("_")
    if len(parts) < 4:
        return None
    try:
        nums = [int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])]
    except ValueError:
        return None
    num123 = f"{nums[0]}_{nums[1]}_{nums[2]}"
    return num123, nums[3]

def group_gt_files(gt_dir: str):
    files = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        files.extend(glob.glob(os.path.join(gt_dir, ext)))
    files = sorted(files)

    groups = {}
    for p in files:
        parsed = parse_name_tokens(Path(p).stem)
        if parsed is None:
            continue
        num123, num4 = parsed
        groups.setdefault(num123, []).append((num4, p))

    final = {}
    for num123, items in groups.items():
        angle_to_path = {}
        for num4, p in sorted(items, key=lambda x: (x[0], x[1])):
            if num4 in TARGET_ANGLES and num4 not in angle_to_path:
                angle_to_path[num4] = p
        if all(a in angle_to_path for a in TARGET_ANGLES):
            final[num123] = [angle_to_path[a] for a in TARGET_ANGLES]
    return final

def stitch_2x2_rgb(paths, image_size=512):
    sub = image_size // 2
    imgs = []
    for p in paths:
        bgr = cv2.imread(p, cv2.IMREAD_COLOR)
        if bgr is None:
            return None
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (sub, sub), interpolation=cv2.INTER_AREA)
        imgs.append(rgb)

    top = np.concatenate([imgs[0], imgs[1]], axis=1)
    bottom = np.concatenate([imgs[2], imgs[3]], axis=1)
    return np.concatenate([top, bottom], axis=0)

def parse_num123_from_output_name(path: str):
    name = Path(path).stem
    m = re.match(r"num123_(-?\d+_-?\d+_-?\d+)_\d+", name)
    return None if m is None else m.group(1)

def rmse(img1, img2):
    d = img1.astype(np.float32) - img2.astype(np.float32)
    return float(np.sqrt(np.mean(d * d)))

if not os.path.isdir(GT_DIR):
    print(f"未找到 GT_DIR: {GT_DIR}")
elif not os.path.isdir(OUT_DIR):
    print(f"未找到 OUT_DIR: {OUT_DIR}")
else:
    grouped_gt = group_gt_files(GT_DIR)
    outs = sorted(glob.glob(os.path.join(OUT_DIR, "*.png")))

    values = []
    skipped = 0

    for op in outs:
        num123 = parse_num123_from_output_name(op)
        if num123 is None or num123 not in grouped_gt:
            skipped += 1
            continue

        gt_rgb = stitch_2x2_rgb(grouped_gt[num123], image_size=512)
        if gt_rgb is None:
            skipped += 1
            continue

        out_bgr = cv2.imread(op, cv2.IMREAD_COLOR)
        if out_bgr is None:
            skipped += 1
            continue
        out_rgb = cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)
        out_rgb = cv2.resize(out_rgb, (gt_rgb.shape[1], gt_rgb.shape[0]), interpolation=cv2.INTER_AREA)

        v = rmse(gt_rgb, out_rgb)
        values.append((op, v))

    if len(values) == 0:
        print("没有可计算的配对样本，请检查路径和命名规则。")
    else:
        rmse_list = [x[1] for x in values]
        print(f"有效样本数: {len(rmse_list)}，跳过: {skipped}")
        print(f"平均 RMSE: {np.mean(rmse_list):.4f} (0-255, 越低越好)")
        print(f"中位数 RMSE: {np.median(rmse_list):.4f} (0-255, 越低越好)")
        print("前 5 个样本:")
        for p, v in values[:5]:
            print(f"  {Path(p).name}: {v:.4f}")

有效样本数: 9，跳过: 0
平均 RMSE: 51.0461 (0-255, 越低越好)
中位数 RMSE: 48.4906 (0-255, 越低越好)
前 5 个样本:
  num123_0_0_0_0001.png: 56.0953
  num123_1_0_0_0002.png: 47.3859
  num123_2_0_0_0003.png: 45.2481
  num123_3_0_0_0004.png: 48.4906
  num123_4_0_0_0005.png: 52.0276


In [2]:
# 点云指标：CD / F1@τ / Completeness / Accuracy / (可选)RMSE
import sys
import subprocess
import numpy as np
from scipy.spatial import cKDTree

# 确保 trimesh 可用
try:
    import trimesh
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "trimesh", "-q"])
    import trimesh

# ===== 选择两份 GLB 点云 =====
REC_GLB = "/home/viplab/liukejia/project/reconstruct/reconstruct_glb_grid4_pos/num123_0_0_0_0001.glb"
GT_GLB  = "/home/viplab/liukejia/project/reconstruct/reconstruct_glb_grid3_pos/num123_0_0_0_0001.glb"

# 采样与阈值设置
MAX_POINTS = 120000
F1_THRESHOLDS = [0.2, 0.5]  # 单位与点云坐标单位一致

def load_points_from_glb(glb_path: str, max_points: int = 120000, seed: int = 0):
    obj = trimesh.load(glb_path, force="scene")
    if isinstance(obj, trimesh.Trimesh):
        pts = np.asarray(obj.vertices, dtype=np.float32)
    else:
        merged = obj.dump(concatenate=True)
        pts = np.asarray(merged.vertices, dtype=np.float32)

    if pts.shape[0] == 0:
        raise ValueError(f"空点云: {glb_path}")

    if max_points is not None and pts.shape[0] > max_points:
        rng = np.random.default_rng(seed)
        idx = rng.choice(pts.shape[0], size=max_points, replace=False)
        pts = pts[idx]
    return pts

def nearest_distances(src_pts: np.ndarray, dst_pts: np.ndarray):
    tree = cKDTree(dst_pts)
    dists, _ = tree.query(src_pts, k=1, workers=-1)
    return dists.astype(np.float64)  # 欧氏距离

def chamfer_distance_l2_squared(p_rec: np.ndarray, p_gt: np.ndarray):
    # CD = E_{p in rec}[min ||p-q||^2] + E_{q in gt}[min ||q-p||^2]
    d_rec = nearest_distances(p_rec, p_gt)
    d_gt = nearest_distances(p_gt, p_rec)
    cd = np.mean(d_rec ** 2) + np.mean(d_gt ** 2)
    return float(cd), d_rec, d_gt

def f1_at_threshold(d_rec: np.ndarray, d_gt: np.ndarray, tau: float):
    # Precision: 重建点中与GT距离<tau的比例
    # Recall: GT点中与重建距离<tau的比例
    precision = float(np.mean(d_rec < tau))
    recall = float(np.mean(d_gt < tau))
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2.0 * precision * recall / (precision + recall)
    return precision, recall, f1

def pointcloud_metrics(p_rec: np.ndarray, p_gt: np.ndarray, thresholds=(0.2, 0.5)):
    cd, d_rec, d_gt = chamfer_distance_l2_squared(p_rec, p_gt)

    # 准确度(Accuracy): rec -> gt 平均最近距离
    accuracy = float(np.mean(d_rec))
    # 完整性(Completeness): gt -> rec 平均最近距离
    completeness = float(np.mean(d_gt))

    # NN 对应下的对称 RMSE（非严格逐点对应，仅作参考）
    rmse_nn = float(np.sqrt((np.mean(d_rec ** 2) + np.mean(d_gt ** 2)) / 2.0))

    out = {
        "cd_l2_squared": cd,
        "accuracy": accuracy,
        "completeness": completeness,
        "rmse_nn": rmse_nn,
        "num_rec": int(p_rec.shape[0]),
        "num_gt": int(p_gt.shape[0]),
        "f1": {},
    }

    for tau in thresholds:
        p, r, f1 = f1_at_threshold(d_rec, d_gt, tau=float(tau))
        out["f1"][float(tau)] = {
            "precision": p,
            "recall": r,
            "f1": f1,
        }
    return out

# ===== 运行 =====
p_rec = load_points_from_glb(REC_GLB, max_points=MAX_POINTS, seed=0)
p_gt = load_points_from_glb(GT_GLB, max_points=MAX_POINTS, seed=1)
m = pointcloud_metrics(p_rec, p_gt, thresholds=F1_THRESHOLDS)

print(f"rec points: {m['num_rec']}, gt points: {m['num_gt']}")
print(f"CD (L2^2): {m['cd_l2_squared']:.6f}")
print(f"Accuracy (rec->gt): {m['accuracy']:.6f}")
print(f"Completeness (gt->rec): {m['completeness']:.6f}")
print(f"RMSE (NN, symmetric): {m['rmse_nn']:.6f}")
for tau, s in m["f1"].items():
    print(
        f"F1@{tau}: P={s['precision']:.4f}, R={s['recall']:.4f}, F1={s['f1']:.4f}"
    )

print("\n说明：若你有严格一一对应点，可再计算严格对应 RMSE。")

rec points: 56, gt points: 56
CD (L2^2): 561897.859141
Accuracy (rec->gt): 561.729914
Completeness (gt->rec): 492.204915
RMSE (NN, symmetric): 530.046158
F1@0.2: P=0.0000, R=0.0000, F1=0.0000
F1@0.5: P=0.0000, R=0.0000, F1=0.0000

说明：若你有严格一一对应点，可再计算严格对应 RMSE。


/tmp/ipykernel_1421502/912153342.py:27: DeprecationWarning: `Scene.dump(concatenate=True)` DEPRECATED FOR REMOVAL APRIL 2025: replace with `Scene.to_geometry()`
  merged = obj.dump(concatenate=True)
